In [1]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("City3D_dataset"):
    subprocess.run(["git","clone","--depth","1","https://github.com/yidahuang/City3D_dataset.git"],check=True)
C3D="/kaggle/working/City3D_dataset"
print(sorted(os.listdir(C3D)))

Cloning into 'City3D_dataset'...


['.git', 'AHN3', 'Dales3d', 'LICENSE', 'README.md', 'Vaihingen']


Updating files: 100% (40788/40788), done.


import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("City3D_dataset"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/yidahuang/City3D_dataset.git"], check=True)
C3D = "/kaggle/working/City3D_dataset"
print(sorted(os.listdir(C3D)))

In [2]:
import os, re, collections
def ply_header(p):
    with open(p,"rb") as f:
        lines=[]
        while True:
            l=f.readline().decode("latin1").strip()
            lines.append(l)
            if l=="end_header" or not l: break
    return lines
def obj_stats(p):
    c=collections.Counter()
    with open(p,errors="ignore") as f:
        for l in f:
            t=l.split(" ",1)[0]
            if t in ("v","f","l"): c[t]+=1
    return dict(c)

for sub in ["AHN3","Dales3d","Vaihingen"]:
    d=f"{C3D}/{sub}"
    if not os.path.isdir(d): print(sub,"MISSING"); continue
    fs=os.listdir(d)
    ext=collections.Counter(os.path.splitext(f)[1].lower() for f in fs)
    ids_ply={os.path.splitext(f)[0] for f in fs if f.endswith(".ply")}
    ids_obj={os.path.splitext(f)[0] for f in fs if f.endswith(".obj")}
    print(f"\n== {sub}: {dict(ext)} | paired={len(ids_ply&ids_obj)} "
          f"ply_only={len(ids_ply-ids_obj)} obj_only={len(ids_obj-ids_ply)}")
    first=sorted(ids_ply&ids_obj)[:1]
    for i in first:
        print("PLY header:", [h for h in ply_header(f"{d}/{i}.ply") if h.startswith(("format","element","property"))])
        print("OBJ counts:", obj_stats(f"{d}/{i}.obj"))


== AHN3: {'.ply': 20039, '.obj': 20039} | paired=20039 ply_only=0 obj_only=0
PLY header: ['format binary_little_endian 1.0', 'element vertex 72', 'property float x', 'property float y', 'property float z', 'property float nx', 'property float ny', 'property float nz', 'property uchar red', 'property uchar green', 'property uchar blue']
OBJ counts: {'v': 79, 'f': 23}

== Dales3d: {'.ply': 300, '.obj': 300} | paired=300 ply_only=0 obj_only=0
PLY header: ['format binary_little_endian 1.0', 'element vertex 596', 'property float x', 'property float y', 'property float z', 'property float nx', 'property float ny', 'property float nz']
OBJ counts: {'v': 171, 'f': 54}

== Vaihingen: {'.ply': 54, '.obj': 54} | paired=54 ply_only=0 obj_only=0
PLY header: ['format binary_little_endian 1.0', 'element vertex 6407', 'property float x', 'property float y', 'property float z', 'property float nx', 'property float ny', 'property float nz']
OBJ counts: {'v': 88, 'f': 25}


In [3]:
import os
for dp, dn, fn in os.walk("/kaggle/input"):
    if dp[len("/kaggle/input"):].count(os.sep) <= 3:
        print(dp, "| files:", len(fn))

/kaggle/input | files: 0


In [4]:
import os, subprocess, json, collections, numpy as np, pandas as pd
SEED=42; N_POINTS=2048; MAX_CORNERS=64; MAX_EDGES=128; MIN_RAW_PTS=16
rng=np.random.default_rng(SEED)
W="/kaggle/working"; os.chdir(W)
if not os.path.exists("City3D_dataset"):
    subprocess.run(["git","clone","--depth","1","https://github.com/yidahuang/City3D_dataset.git"],check=True)
C3D=f"{W}/City3D_dataset"; OUT=f"{W}/phase6_city3d"; os.makedirs(OUT,exist_ok=True)
SUBSETS=[s for s in ["AHN3","Dales3d","Vaihingen"] if os.path.isdir(f"{C3D}/{s}")]

_T={"char":"i1","uchar":"u1","short":"i2","ushort":"u2","int":"i4","uint":"u4",
    "float":"f4","double":"f8","int8":"i1","uint8":"u1","int16":"i2","uint16":"u2",
    "int32":"i4","uint32":"u4","float32":"f4","float64":"f8"}

def read_ply_xyz(p):
    with open(p,"rb") as f:
        hdr=[]
        while True:
            l=f.readline().decode("latin1").strip(); hdr.append(l)
            if l=="end_header": break
        fmt=[h for h in hdr if h.startswith("format")][0].split()[1]
        n=None; props=[]; cur=None
        for h in hdr:
            t=h.split()
            if t[:1]==["element"]:
                cur=t[1]
                if cur=="vertex": n=int(t[2])
            elif t[:1]==["property"] and cur=="vertex" and t[1]!="list":
                props.append((t[2],t[1]))
        if fmt=="ascii":
            a=np.loadtxt(f,max_rows=n,ndmin=2); names=[q[0] for q in props]
            xyz=a[:,[names.index("x"),names.index("y"),names.index("z")]]
        else:
            end="<" if fmt=="binary_little_endian" else ">"
            dt=np.dtype([(nm,end+_T[tp]) for nm,tp in props])
            a=np.frombuffer(f.read(dt.itemsize*n),dtype=dt,count=n)
            xyz=np.stack([a["x"],a["y"],a["z"]],1).astype("f8")
    return xyz

def read_obj(p):
    v=[];faces=[];lines=[]
    for l in open(p,errors="ignore"):
        t=l.split()
        if not t: continue
        if t[0]=="v": v.append([float(x) for x in t[1:4]])
        elif t[0]=="f": faces.append([int(x.split("/")[0])-1 for x in t[1:]])
        elif t[0]=="l": lines.append([int(x)-1 for x in t[1:3]])
    return np.array(v,dtype="f8").reshape(-1,3),faces,lines

print(SUBSETS)

['AHN3', 'Dales3d', 'Vaihingen']


In [5]:
import os, numpy as np
C3D="/kaggle/working/City3D_dataset"
for s in ["AHN3","Dales3d","Vaihingen"]:
    d=f"{C3D}/{s}"; i=sorted(x[:-4] for x in os.listdir(d) if x.endswith(".ply"))[0]
    p=read_ply_xyz(f"{d}/{i}.ply"); v,fc,ln=read_obj(f"{d}/{i}.obj")
    print(s,i,"pts xyz min",p.min(0).round(2),"max",p.max(0).round(2),"| obj min",v.min(0).round(2),"max",v.max(0).round(2))

AHN3 00000 pts xyz min [46.2  81.87 -5.82] max [50.43 84.9  -3.49] | obj min [45.96 81.44 -5.98] max [50.66 85.03 -3.58]
Dales3d 00000 pts xyz min [412.18  76.58   6.08] max [416.59  81.34   8.98] | obj min [412.02  76.48   1.05] max [416.65  81.48   8.99]
Vaihingen 00000 pts xyz min [4.9715094e+05 5.4193575e+06 2.6093000e+02] max [4.9716484e+05 5.4194045e+06 2.7855000e+02] | obj min [4.97150820e+05 5.41935708e+06 2.65000000e+02] max [4.97165000e+05 5.41940473e+06 2.77750000e+02]


In [6]:
import torch, shutil
from scipy.spatial import ConvexHull
GROUND_TOL=0.3; SHARD=256
MAX_C_STORE=512; MAX_E_STORE=1024
SH=f"{OUT}/shards"; shutil.rmtree(SH,ignore_errors=True); os.makedirs(SH)

def roof_reference(v,faces):
    keep=np.where(v[:,2]>v[:,2].min()+GROUND_TOL)[0]
    m={o:i for i,o in enumerate(keep)}; E=set()
    for f in faces:
        for k in range(len(f)):
            a,b=f[k],f[(k+1)%len(f)]
            if a in m and b in m and a!=b: E.add((min(m[a],m[b]),max(m[a],m[b])))
    return v[keep],np.array(sorted(E),dtype=int).reshape(-1,2)
def density(pts):
    try: a=ConvexHull(pts[:,:2]).volume
    except Exception: a=np.ptp(pts[:,0])*np.ptp(pts[:,1])
    return len(pts)/max(a,1e-6),a

rows=[];rej=[];buf=[];sh=0
def flush():
    global buf,sh
    if buf: torch.save(buf,f"{SH}/shard_{sh:04d}.pt"); buf=[]; sh+=1
for s in SUBSETS:
    d=f"{C3D}/{s}"
    ids=sorted({x[:-4] for x in os.listdir(d) if x.endswith(".ply")}&{x[:-4] for x in os.listdir(d) if x.endswith(".obj")})
    for n,i in enumerate(ids):
        bid=f"{s.lower()}_{i}"
        try:
            pts=read_ply_xyz(f"{d}/{i}.ply"); v,fc,_=read_obj(f"{d}/{i}.obj")
            if not (np.isfinite(pts).all() and np.isfinite(v).all()): raise ValueError("nonfinite")
            if len(pts)<MIN_RAW_PTS: raise ValueError("few_points")
            rv,E=roof_reference(v,fc)
            if len(rv)<4 or len(E)<4: raise ValueError("too_few_roof_vertices_or_edges")
            if len(rv)>MAX_C_STORE or len(E)>MAX_E_STORE: raise ValueError("too_complex_for_store")
            in_scope=(len(rv)<=MAX_CORNERS) and (len(E)<=MAX_EDGES)
            dens,area=density(pts)
            c=pts.mean(0); sc=float(np.abs(pts-c).max()) or 1.0
            p=((pts-c)/sc); p=p[rng.choice(len(p),N_POINTS,replace=len(p)<N_POINTS)]
            cp=np.zeros((MAX_C_STORE,3),"f4"); cp[:len(rv)]=(rv-c)/sc
            ep=np.full((MAX_E_STORE,2),-1,"i4"); ep[:len(E)]=E
            rows.append(dict(building_id=bid,subset=s,shard=sh,row=len(buf),raw_points=len(pts),
                n_corners=len(rv),n_edges=len(E),in_model_scope=in_scope,density_pts_m2=dens,
                xy_area_m2=area,cx=c[0],cy=c[1],cz=c[2],scale=sc))
            buf.append(dict(building_id=bid,points=p.astype("f4"),corners=cp,edges=ep,
                n_corners=len(rv),n_edges=len(E)))
            if len(buf)==SHARD: flush()
        except Exception as e: rej.append(dict(building_id=bid,subset=s,reason=str(e)))
        if n%2000==0: print(s,n,"/",len(ids))
flush()
idx=pd.DataFrame(rows); rj=pd.DataFrame(rej)
print(idx.groupby(["subset","in_model_scope"]).size())
print(rj.groupby(["subset","reason"]).size() if len(rj) else "no rejects")
assert len(idx)>0 and idx.building_id.is_unique

AHN3 0 / 20039
AHN3 2000 / 20039
AHN3 4000 / 20039
AHN3 6000 / 20039
AHN3 8000 / 20039
AHN3 10000 / 20039
AHN3 12000 / 20039
AHN3 14000 / 20039
AHN3 16000 / 20039
AHN3 18000 / 20039
AHN3 20000 / 20039
Dales3d 0 / 300
Vaihingen 0 / 54
subset     in_model_scope
AHN3       False             10293
           True               8436
Dales3d    False               125
           True                 28
Vaihingen  False                27
           True                 24
dtype: int64
subset     reason                        
AHN3       too_complex_for_store             487
           too_few_roof_vertices_or_edges    823
Dales3d    too_complex_for_store             147
Vaihingen  too_complex_for_store               3
dtype: int64


In [7]:
thr=float(idx.density_pts_m2.median())   # placeholder: replace with Tallinn threshold
idx["coverage_class"]=np.where(idx.density_pts_m2>=thr,"dense","sparse")
idx.to_csv(f"{OUT}/city3d_index.csv",index=False); rj.to_csv(f"{OUT}/city3d_rejected.csv",index=False)
json.dump(dict(seed=SEED,n_points=N_POINTS,max_corners=MAX_CORNERS,max_edges=MAX_EDGES,
  ground_tol_m=GROUND_TOL,shard_size=SHARD,density_threshold=thr,
  counts=idx.subset.value_counts().to_dict(),n_rejected=len(rj)),open(f"{OUT}/city3d_cache_config.json","w"),indent=2)

r=idx.iloc[0]; S=torch.load(f"{SH}/shard_{int(r.shard):04d}.pt",weights_only=False)[int(r.row)]
assert S["building_id"]==r.building_id and S["points"].shape==(N_POINTS,3)
assert np.abs(S["points"]).max()<=1.0+1e-6
back=S["corners"][:S["n_corners"]]*r.scale+np.array([r.cx,r.cy,r.cz])
print("restored corner z range:",back[:,2].min().round(2),back[:,2].max().round(2))
n_shard=len([f for f in os.listdir(SH) if f.endswith(".pt")])
assert sum(len(torch.load(f"{SH}/{f}",weights_only=False)) for f in os.listdir(SH))==len(idx)
print("OK shards:",n_shard,"buildings:",len(idx))

restored corner z range: -3.61 -3.58
OK shards: 74 buildings: 18933


In [8]:
print(rj.groupby(["subset","reason"]).size())
print(idx.groupby("subset").size())

import random
samp=[]
for s in SUBSETS:
    d=f"{C3D}/{s}"; ids=sorted(x[:-4] for x in os.listdir(d) if x.endswith(".obj"))
    samp+= [(s,i) for i in random.Random(0).sample(ids,min(150,len(ids)))]
res=[]
for tol in [0.1,0.3,0.5,1.0]:
    for s,i in samp:
        v,fc,_=read_obj(f"{C3D}/{s}/{i}.obj")
        keep=np.where(v[:,2]>v[:,2].min()+tol)[0]; m=set(keep.tolist())
        E={(min(a,b),max(a,b)) for f in fc for k in range(len(f))
           for a,b in [(f[k],f[(k+1)%len(f)])] if a in m and b in m and a!=b}
        res.append(dict(tol=tol,subset=s,nv=len(keep),ne=len(E),total_v=len(v)))
R=pd.DataFrame(res)
R["ok"]=(R["nv"].between(4,MAX_CORNERS))&(R["ne"].between(4,MAX_EDGES))
print(R.groupby(["tol","subset"])["ok"].mean().unstack().round(2))
print(R[R["tol"]==0.3].groupby("subset")[["total_v","nv","ne"]].describe().round(1).T)

subset     reason                        
AHN3       too_complex_for_store             487
           too_few_roof_vertices_or_edges    823
Dales3d    too_complex_for_store             147
Vaihingen  too_complex_for_store               3
dtype: int64
subset
AHN3         18729
Dales3d        153
Vaihingen       51
dtype: int64
subset  AHN3  Dales3d  Vaihingen
tol                             
0.1     0.49     0.07       0.44
0.3     0.47     0.07       0.44
0.5     0.45     0.07       0.43
1.0     0.43     0.07       0.43
subset           AHN3  Dales3d  Vaihingen
total_v count   150.0    150.0       54.0
        mean    204.7    769.0      220.1
        std     296.3    580.6      243.9
        min      24.0     56.0       24.0
        25%      68.0    263.5       79.2
        50%     117.5    710.0      123.0
        75%     264.8   1045.0      282.8
        max    2752.0   2807.0     1122.0
nv      count   150.0    150.0       54.0
        mean    139.9    636.1      151.0
        std 